In [1]:
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from mask_dataset import WheatBinaryDataset

C:\Users\lochana.marasingha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda"

# Training Config
train_transform = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])


In [6]:
dataset = WheatBinaryDataset('masks/train', transform=train_transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)


Pre-loading images into RAM for speed...


In [7]:
model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1, # Binary segmentation
).to(DEVICE)


In [8]:
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
model.train()
for epoch in range(100):
    epoch_loss = 0
    print(f"\n--- Starting Epoch {epoch+1} ---", flush=True)

    for batch_idx, (images, masks) in enumerate(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Print every 5 batches to show signs of life
        if batch_idx % 5 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}", flush=True)

    avg_loss = epoch_loss / len(train_loader)
    print(f"--- Epoch {epoch+1} Finished | Avg Loss: {avg_loss:.4f} ---", flush=True)


--- Starting Epoch 1 ---
Epoch 1 | Batch 0/73 | Loss: 0.3861
